In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp

import skewerjax.summary_stats.toy_model_samplers as toy_model_samplers
import skewerjax.summary_stats.xi1D as xi1D

In [ ]:
x_a, x_b = 0.0, 120.0

sampler_i = toy_model_samplers.SimplePeriodicFluxSampler(
    x_a=x_a, x_b=x_b, n_x=2400, periodic=False, period=20.0, width=1.2, depth=0.99, phase=None, jitter_pos=1.0, jitter_width_frac=0.5
)

sampler_j = toy_model_samplers.SimplePeriodicFluxSampler(
    x_a=x_a, x_b=x_b, n_x=2400, periodic=False, period=30.0, width=0.3, depth=0.99, phase=None, jitter_pos=3.0, jitter_width_frac=0.0
)

In [ ]:
# Plot multiple realizations (different seeds) as stacked rows.
# Each row shows Fi^{(s)}(x) (black) and Fj^{(s)}(x) (red).

import matplotlib.pyplot as plt

# choose which realizations (seeds) to display
seeds = [0, 1, 2]   # <-- edit as you like
n_rows = len(seeds)

fig, axes = plt.subplots(
    n_rows, 1,
    figsize=(12, 2.6 * n_rows),
    sharex=True,
    sharey=True,
)

if n_rows == 1:
    axes = [axes]

for row, s in enumerate(seeds):
    ax = axes[row]

    x, Fi = sampler_i.sample(seed=s)
    _, Fj = sampler_j.sample(seed=s)

    ax.plot(x, Fi, color="k", lw=2.0, label=r"$F_i^{(s)}(x)$" if row == 0 else None)
    ax.plot(x, Fj, color="crimson", lw=2.0, alpha=0.9, label=r"$F_j^{(s)}(x)$" if row == 0 else None)

    ax.axvline(x_a, color="k", lw=1.5, ls="--", alpha=0.35)
    ax.axvline(x_b, color="k", lw=1.5, ls="--", alpha=0.35)

    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel(r"$F$")

    # small label indicating realization index/seed
    ax.text(
        0.02, 0.90, rf"$s={s}$",
        transform=ax.transAxes, ha="left", va="top", fontsize=11, bbox=dict(boxstyle="round", facecolor="white", edgecolor="k", alpha=0.85),
    )

# shared x-label on bottom only
axes[-1].set_xlabel(r"$x$")

# single legend (top panel)
axes[0].legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
delta_sampler_i = toy_model_samplers.FractionalFluctuationFieldSampler(sampler_i, periodic_integration=False)
delta_sampler_j = toy_model_samplers.FractionalFluctuationFieldSampler(sampler_j, periodic_integration=False)

x, delta_i, F_i, meanFi = delta_sampler_i.sample(seed=0, return_flux=True, return_mean=True)
x, delta_j, F_j, meanFj = delta_sampler_j.sample(seed=0, return_flux=True, return_mean=True)

In [ ]:
# choose which realizations (seeds) to display
seeds = [0, 1, 2]   # <-- edit as you like
n_rows = len(seeds)

fig, axes = plt.subplots(
    n_rows, 1,
    figsize=(12, 2.6 * n_rows),
    sharex=True,
    sharey=True,
)

if n_rows == 1:
    axes = [axes]

# First pass: compute common y-limits across all shown realizations (robust)
all_vals = []
for s in seeds:
    x, di = delta_sampler_i.sample(seed=s)
    _, dj = delta_sampler_j.sample(seed=s)
    all_vals.append(di)
    all_vals.append(dj)

all_vals = plt.np.concatenate(all_vals) if hasattr(plt, "np") else __import__("numpy").concatenate(all_vals)

for row, s in enumerate(seeds):
    ax = axes[row]

    x, di, Fi, meanFi = delta_sampler_i.sample(seed=s, return_flux=True, return_mean=True)
    _, dj, Fj, meanFj = delta_sampler_j.sample(seed=s, return_flux=True, return_mean=True)

    ax.plot(x, di, color="k", lw=2.0, label=r"$\delta_i^{(s)}(x)$" if row == 0 else None)
    ax.plot(x, dj, color="crimson", lw=2.0, alpha=0.9, label=r"$\delta_j^{(s)}(x)$" if row == 0 else None)

    ax.axvline(x_a, color="k", lw=1.5, ls="--", alpha=0.35)
    ax.axvline(x_b, color="k", lw=1.5, ls="--", alpha=0.35)
    ax.set_ylabel(r"$\delta$")

    ax.axhline(0.0, color="k", lw=1.5, ls=":", alpha=0.5)

    # small label indicating realization index/seed + the per-realization means used in δ definition
    ax.text(
        0.02, 0.90, rf"$s={s}$" + "\n" + rf"$\langle F_i\rangle={meanFi:.3f}$" + "\n" + rf"$\langle F_j\rangle={meanFj:.3f}$",
        transform=ax.transAxes, ha="left", va="top", fontsize=11, bbox=dict(boxstyle="round", facecolor="white", edgecolor="k", alpha=0.85),
    )

# shared x-label on bottom only
axes[-1].set_xlabel(r"$x$")

# single legend (top panel)
axes[0].legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
def is_almost_uniform(x: np.ndarray, rtol: float = 1e-3, atol: float = 1e-8) -> bool:
    dx = np.diff(x)
    return np.allclose(dx, np.mean(dx), rtol=rtol, atol=atol)


def stack_deltas(delta_sampler, seeds, return_flux=False, return_mean=False):
    """Collect (x, delta[s], F[s], meanF[s]) for multiple seeds and stack deltas into (N_samples, N_x)."""
    deltas = []
    means = []
    fluxes = []

    x_ref = None
    for s in seeds:
        out = delta_sampler.sample(seed=s, return_flux=return_flux, return_mean=return_mean)
        if return_flux and return_mean:
            x, d, F, mF = out
            fluxes.append(np.asarray(F))
            means.append(float(mF))
        elif return_flux:
            x, d, F = out
            fluxes.append(np.asarray(F))
        elif return_mean:
            x, d, mF = out
            means.append(float(mF))
        else:
            x, d = out

        x = np.asarray(x)
        d = np.asarray(d)
        if x_ref is None:
            x_ref = x
        else:
            if x.shape != x_ref.shape or not np.allclose(x, x_ref):
                raise ValueError("All realizations must share the same x grid for this demo.")

        deltas.append(d)

    deltas = np.stack(deltas, axis=0)  # (N_samples, N_x)
    fluxes = np.stack(fluxes, axis=0) if len(fluxes) > 0 else None
    means = np.asarray(means) if len(means) > 0 else None
    return x_ref, deltas, fluxes, means


def demo_2pcf_concepts(
    sampler_i,
    sampler_j,
    *,
    seeds_show=(0, 1, 2),
    seeds_ens=tuple(range(0, 64)),
    n_bins=64,
    r_min=0.0,
    r_max=None,
    include_self=False,
    chunk_size=1 << 15,
    center=False,          # δ already ~mean-zero if mean flux is spatial average
    standardize=False,     # keep physical amplitude by default
    outdir=None,
):
    # -------------------------
    # 0) Build the δ samplers
    # -------------------------
    delta_sampler_i = toy_model_samplers.FractionalFluctuationFieldSampler(sampler_i, periodic_integration=False)
    delta_sampler_j = toy_model_samplers.FractionalFluctuationFieldSampler(sampler_j, periodic_integration=False)

    # -------------------------
    # 1) Realizations of δ(x)
    # -------------------------
    x, di_show, Fi_show, meanFi_show = stack_deltas(delta_sampler_i, seeds_show, return_flux=True, return_mean=True)
    _, dj_show, Fj_show, meanFj_show = stack_deltas(delta_sampler_j, seeds_show, return_flux=True, return_mean=True)

    x_a, x_b = float(np.min(x)), float(np.max(x))
    L = x_b - x_a

    fig, axes = plt.subplots(len(seeds_show), 1, figsize=(12, 2.7 * len(seeds_show)), sharex=True, sharey=True)
    if len(seeds_show) == 1:
        axes = [axes]

    # common y-limits for δ across displayed seeds
    all_vals = np.concatenate([di_show.ravel(), dj_show.ravel()])
    y_lo, y_hi = np.nanpercentile(all_vals, [1.0, 99.0])
    pad = 0.10 * (y_hi - y_lo) if y_hi > y_lo else 1.0

    for row, s in enumerate(seeds_show):
        ax = axes[row]
        ax.plot(x, di_show[row], color="k", lw=2.0, label=r"$\delta_i^{(s)}(x)$" if row == 0 else None)
        ax.plot(x, dj_show[row], color="crimson", lw=2.0, alpha=0.9, label=r"$\delta_j^{(s)}(x)$" if row == 0 else None)

        ax.axhline(0.0, color="k", lw=1.2, ls=":", alpha=0.5)
        ax.set_ylim(y_lo - pad, y_hi + pad)
        ax.set_ylabel(r"$\delta$")

        ax.text(
            0.02, 0.90,
            rf"$s={s}$"
            + "\n" + rf"$\langle F_i\rangle={meanFi_show[row]:.3f}$"
            + "\n" + rf"$\langle F_j\rangle={meanFj_show[row]:.3f}$"
            + "\n" + rf"$\langle\delta_i\rangle={np.mean(di_show[row]):+.2e}$"
            + "\n" + rf"$\langle\delta_j\rangle={np.mean(dj_show[row]):+.2e}$",
            transform=ax.transAxes, ha="left", va="top",
            fontsize=10,
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="k", alpha=0.85),
        )

    axes[-1].set_xlabel(r"$x$")
    axes[0].legend(loc="upper right", frameon=True)
    fig.suptitle("Realization-level fluctuation fields $\\delta_i^{(s)}(x)$ and $\\delta_j^{(s)}(x)$", y=0.995)
    fig.tight_layout()

    # -------------------------
    # 2) Precompute pb (geometry/window)
    # -------------------------
    pb = xi1D.precompute_pair_binning_1d(
        jnp.asarray(x),
        n_bins=n_bins,
        r_min=r_min,
        r_max=(L if r_max is None else r_max),
        include_self=include_self,
        chunk_size=chunk_size,
    )
    r_centers = np.asarray(pb.r_centers)
    counts = np.asarray(pb.counts_per_bin)
    den = np.asarray(pb.den_per_bin)

    # window / denominator visualization
    fig2, (axc, axd) = plt.subplots(2, 1, figsize=(12, 6.0), sharex=True, gridspec_kw={"hspace": 0.08})
    axc.plot(r_centers, counts, lw=2.0, color="k")
    axc.set_ylabel("pair counts per bin")

    axd.plot(r_centers, den, lw=2.0, color="k")
    axd.set_ylabel("denominator $\\sum w_{ij}$ per bin")
    axd.set_xlabel(r"$r$")
    axc.set_title("Discrete window/normalization: fewer valid pairs at large separations")

    # If x is uniform, compare counts to (N - lag) scaling (discrete analogue of L-r)
    if is_almost_uniform(x):
        dx = float(np.mean(np.diff(x)))
        # crude expected trend ~ (L-r) up to normalization; show as a dashed guide
        guide = (L - r_centers)
        guide = guide / np.max(guide) * np.max(counts)
        axc.plot(r_centers, guide, ls="--", lw=1.8, color="gray", alpha=0.8, label=r"guide $\propto (L-r)$")
        axc.legend(frameon=True)

    fig2.tight_layout()

    # -------------------------
    # 3) Per-sample 2PCFs (auto + cross) for shown seeds
    # -------------------------
    # Use your JAX routines directly on stacked arrays (N_samples, N_x)
    xi_ii_show = np.asarray(
        xi1D.two_point_corr_1d_from_precomp(pb, jnp.asarray(di_show), center=center, standardize=standardize)
    )
    xi_jj_show = np.asarray(
        xi1D.two_point_corr_1d_from_precomp(pb, jnp.asarray(dj_show), center=center, standardize=standardize)
    )

    # cross: sym and the two one-orientation versions
    xi_ij_sym_show = np.asarray(
        xi1D.two_point_corr_1d_cross_from_precomp(pb, jnp.asarray(di_show), jnp.asarray(dj_show),
                                                  center=center, standardize=standardize, symmetrize=True)
    )
    xi_ij_one_show = np.asarray(
        xi1D.two_point_corr_1d_cross_from_precomp(pb, jnp.asarray(di_show), jnp.asarray(dj_show),
                                                  center=center, standardize=standardize, symmetrize=False)
    )
    xi_ji_one_show = np.asarray(
        xi1D.two_point_corr_1d_cross_from_precomp(pb, jnp.asarray(dj_show), jnp.asarray(di_show),
                                                  center=center, standardize=standardize, symmetrize=False)
    )

    # identity check: sym ≈ one + swapped-one
    sym_resid = xi_ij_sym_show - (xi_ij_one_show + xi_ji_one_show)

    fig3, axes3 = plt.subplots(3, 1, figsize=(12, 9.0), sharex=True, gridspec_kw={"hspace": 0.10})
    axA, axB, axC = axes3

    # Plot per-sample curves for a few seeds (one curve per shown seed)
    for k, s in enumerate(seeds_show):
        axA.plot(r_centers, xi_ii_show[k], lw=2.0, alpha=0.85)
        axA.plot(r_centers, xi_jj_show[k], lw=2.0, alpha=0.85, ls="--")

    axA.axhline(0.0, color="k", lw=1.0, alpha=0.3)
    axA.set_ylabel(r"$\widehat{\xi}(r)$")
    axA.set_title("Per-sample (realization-level) 2PCFs (spatial averages)")

    for k, s in enumerate(seeds_show):
        axB.plot(r_centers, xi_ij_one_show[k], lw=2.0, alpha=0.85)
        axB.plot(r_centers, xi_ij_sym_show[k], lw=2.0, alpha=0.85, ls="--")

    axB.axhline(0.0, color="k", lw=1.0, alpha=0.3)
    axB.set_ylabel(r"$\widehat{\xi}_{ij}(r)$")

    for k, s in enumerate(seeds_show):
        axC.plot(r_centers, sym_resid[k], lw=2.0, alpha=0.85)

    axC.axhline(0.0, color="k", lw=1.0, alpha=0.6)
    axC.set_ylabel("residual")
    axC.set_xlabel(r"$r$")
    axC.set_title("Symmetrization identity check (should be ~0)")

    axA.legend(frameon=True, fontsize=9, ncol=2)
    axB.legend(frameon=True, fontsize=9, ncol=2)
    axC.legend(frameon=True, fontsize=9, ncol=2)
    fig3.tight_layout()

    # -------------------------
    # 4) Ensemble estimator: average over many realizations
    # -------------------------
    x_ens, di_ens, _, _ = stack_deltas(delta_sampler_i, seeds_ens, return_flux=False, return_mean=False)
    _, dj_ens, _, _ = stack_deltas(delta_sampler_j, seeds_ens, return_flux=False, return_mean=False)

    # sanity: use same x
    assert np.allclose(x_ens, x)

    xi_ii_ens = np.asarray(xi1D.two_point_corr_1d_from_precomp(pb, jnp.asarray(di_ens), center=center, standardize=standardize))
    xi_jj_ens = np.asarray(xi1D.two_point_corr_1d_from_precomp(pb, jnp.asarray(dj_ens), center=center, standardize=standardize))
    xi_ij_ens = np.asarray(xi1D.two_point_corr_1d_cross_from_precomp(pb, jnp.asarray(di_ens), jnp.asarray(dj_ens),
                                                                      center=center, standardize=standardize, symmetrize=True))

    # "ensemble estimate" is mean over samples
    mean_ii = np.nanmean(xi_ii_ens, axis=0)
    std_ii  = np.nanstd(xi_ii_ens, axis=0)

    mean_jj = np.nanmean(xi_jj_ens, axis=0)
    std_jj  = np.nanstd(xi_jj_ens, axis=0)

    mean_ij = np.nanmean(xi_ij_ens, axis=0)
    std_ij  = np.nanstd(xi_ij_ens, axis=0)

    fig4, ax4 = plt.subplots(figsize=(12, 5.5))
    ax4.plot(r_centers, mean_ii, color="k", lw=2.2)
    ax4.fill_between(r_centers, mean_ii - std_ii, mean_ii + std_ii, color="k", alpha=0.10)

    ax4.plot(r_centers, mean_jj, color="crimson", lw=2.2, label=r"$\widehat{\xi}_{jj}(r)$")
    ax4.fill_between(r_centers, mean_jj - std_jj, mean_jj + std_jj, color="crimson", alpha=0.10)

    ax4.plot(r_centers, mean_ij, color="royalblue", lw=2.2, label=r"$\widehat{\xi}_{ij,\mathrm{sym}}(r)$")
    ax4.fill_between(r_centers, mean_ij - std_ij, mean_ij + std_ij, color="royalblue", alpha=0.10)

    ax4.axhline(0.0, color="k", lw=1.0, alpha=0.35)
    ax4.set_xlabel(r"$r$")
    ax4.set_ylabel(r"$\xi(r)$")
    ax4.set_title(f"Ensemble estimate from {len(seeds_ens)} realizations (mean ± std across realizations)")
    ax4.legend(frameon=True, fontsize=10)
    fig4.tight_layout()

    # -------------------------
    # 5) Convergence with N (visual + quantitative)
    # -------------------------
    # Use the largest-N mean as a reference "truth" for demonstration
    ref = mean_ij.copy()

    Ns = [1, 2, 4, 8, 16, 32, len(seeds_ens)]
    Ns = [N for N in Ns if N <= len(seeds_ens)]
    rmses = []

    fig5, ax5 = plt.subplots(figsize=(12, 5.5))
    for N in Ns:
        estN = np.nanmean(xi_ij_ens[:N], axis=0)
        ax5.plot(r_centers, estN, lw=2.0, alpha=0.85, label=rf"$N={N}$")

        rmse = np.sqrt(np.nanmean((estN - ref) ** 2))
        rmses.append(rmse)

    ax5.plot(r_centers, ref, color="k", lw=3.0, alpha=0.9, label=rf"reference ($N={len(seeds_ens)}$)")
    ax5.axhline(0.0, color="k", lw=1.0, alpha=0.25)
    ax5.set_xlabel(r"$r$")
    ax5.set_ylabel(r"$\widehat{\xi}_{ij,\mathrm{sym}}(r)$")
    ax5.set_title("Convergence of the ensemble estimator as N increases")
    ax5.legend(frameon=True, fontsize=9, ncol=2)
    fig5.tight_layout()

    fig6, ax6 = plt.subplots(figsize=(7.5, 4.8))
    ax6.plot(Ns, rmses, marker="o", lw=2.0)
    ax6.set_xlabel(r"$N$")
    ax6.set_ylabel("RMSE vs reference")
    ax6.set_title("Quantitative convergence (Monte Carlo)")
    ax6.grid(True, alpha=0.25)
    fig6.tight_layout()

    # -------------------------
    # 6) Optionally save figures
    # -------------------------
    if outdir is not None:
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, "01_realizations_delta.png"), dpi=200, bbox_inches="tight")
        fig2.savefig(os.path.join(outdir, "02_window_denominator.png"), dpi=200, bbox_inches="tight")
        fig3.savefig(os.path.join(outdir, "03_per_sample_2pcf_and_symmetry.png"), dpi=200, bbox_inches="tight")
        fig4.savefig(os.path.join(outdir, "04_ensemble_estimate.png"), dpi=200, bbox_inches="tight")
        fig5.savefig(os.path.join(outdir, "05_convergence_curves.png"), dpi=200, bbox_inches="tight")
        fig6.savefig(os.path.join(outdir, "06_convergence_rmse.png"), dpi=200, bbox_inches="tight")

    plt.show()

In [ ]:
demo_2pcf_concepts(
    sampler_i, sampler_j,
    seeds_show=(0, 1, 2),
    seeds_ens=tuple(range(0, 128)),
    n_bins=80,
    r_min=0.0,
    r_max=None,          # defaults to L if you pass None here
    include_self=False,
    chunk_size=1 << 15,
    center=False,
    standardize=False,
    outdir=None,
)